Here we will create a seperate ALS for each membership
- genre
- track
- album
- artist

and give a weighted score towards each

In [1]:
import os
import sys
import time
import subprocess
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.recommendation import ALS

In [2]:
jdk_path = r"C:\Program Files\Java\jdk-17"
os.environ["JAVA_HOME"] = jdk_path
os.environ["PATH"] = jdk_path + r"\bin;" + os.environ["PATH"]

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("MultiALS_MusicRecommender")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.default.parallelism", "64")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark started successfully:", spark.version)

Spark started successfully: 3.5.0


In [3]:
# Cell 2: Load CSV files

train_df = (
    spark.read.csv("Assets/CSV/train_data.csv", header=True, inferSchema=True)
    .select(
        F.col("UserID").cast(IntegerType()).alias("userID"),
        F.col("ItemID").cast(IntegerType()).alias("itemID"),
        F.col("Rating").cast(FloatType()).alias("rating")
    )
)

test_df = (
    spark.read.csv("Assets/CSV/test_data.csv", header=True, inferSchema=True)
    .select(
        F.col("UserID").cast(IntegerType()).alias("userID"),
        F.col("TrackID").cast(IntegerType()).alias("trackID")
    )
)

track_df = (
    spark.read.csv("Assets/CSV/track_data.csv", header=True, inferSchema=True)
)

album_df = (
    spark.read.csv("Assets/CSV/album_data.csv", header=True, inferSchema=True)
    .select(F.col("AlbumID").cast(IntegerType()).alias("albumID"))
)

artist_df = (
    spark.read.csv("Assets/CSV/artist_data.csv", header=True, inferSchema=True)
    .select(F.col("ArtistID").cast(IntegerType()).alias("artistID"))
)

genre_df = (
    spark.read.csv("Assets/CSV/genre_data.csv", header=True, inferSchema=True)
    .select(F.col("GenreID").cast(IntegerType()).alias("genreID"))
)


In [4]:
# Cell 3: Build track metadata tables

# Standardize core columns
track_meta_df = (
    track_df
    .select(
        F.col("TrackID").cast(IntegerType()).alias("trackID"),
        F.col("AlbumID").cast(IntegerType()).alias("albumID"),
        F.col("ArtistID").cast(IntegerType()).alias("artistID"),
        *[F.col(c).cast(IntegerType()).alias(c) for c in track_df.columns if c.startswith("Genre")]
    )
)

genre_cols = [c for c in track_meta_df.columns if c.startswith("Genre")]

# Explode genres into long format: one row per (trackID, genreID)
track_genres_df = (
    track_meta_df
    .select(
        "trackID",
        F.explode_outer(
            F.array(*[F.col(c) for c in genre_cols])
        ).alias("genreID")
    )
    .filter(F.col("genreID").isNotNull())
    .dropDuplicates(["trackID", "genreID"])
)

# Keep only the track -> album/artist mapping here
track_core_df = (
    track_meta_df
    .select("trackID", "albumID", "artistID")
    .dropDuplicates(["trackID"])
)

print("track_core_df")
track_core_df.show(5, truncate=False)

print("track_genres_df")
track_genres_df.show(10, truncate=False)


track_core_df
+-------+-------+--------+
|trackID|albumID|artistID|
+-------+-------+--------+
|38     |NULL   |NULL    |
|273    |264415 |206388  |
|300    |174302 |239122  |
|475    |76423  |6756    |
|585    |261087 |226368  |
+-------+-------+--------+
only showing top 5 rows

track_genres_df
+-------+-------+
|trackID|genreID|
+-------+-------+
|27     |173467 |
|88     |146792 |
|116    |279143 |
|120    |8400   |
|224    |33204  |
|257    |176858 |
|332    |208198 |
|338    |34486  |
|407    |196528 |
|485    |146792 |
+-------+-------+
only showing top 10 rows



In [5]:
# Cell 4: Build entity ID lookup tables

track_ids_df = track_core_df.select(F.col("trackID").alias("itemID")).dropDuplicates()
album_ids_df = album_df.select(F.col("albumID").alias("itemID")).dropDuplicates()
artist_ids_df = artist_df.select(F.col("artistID").alias("itemID")).dropDuplicates()
genre_ids_df = genre_df.select(F.col("genreID").alias("itemID")).dropDuplicates()

print("Unique track IDs :", track_ids_df.count())
print("Unique album IDs :", album_ids_df.count())
print("Unique artist IDs:", artist_ids_df.count())
print("Unique genre IDs :", genre_ids_df.count())


Unique track IDs : 224041
Unique album IDs : 52829
Unique artist IDs: 18674
Unique genre IDs : 567


In [6]:
# Cell 5: Create clean ALS training tables
# Split the data

# Track ratings
train_track_df = (
    train_df.join(track_ids_df, on="itemID", how="inner")
    .select("userID", "itemID", "rating")
)

# Album ratings
train_album_df = (
    train_df.join(album_ids_df, on="itemID", how="inner")
    .select("userID", "itemID", "rating")
)

# Artist ratings
train_artist_df = (
    train_df.join(artist_ids_df, on="itemID", how="inner")
    .select("userID", "itemID", "rating")
)

# Genre ratings
train_genre_df = (
    train_df.join(genre_ids_df, on="itemID", how="inner")
    .select("userID", "itemID", "rating")
)

print("track ratings :", train_track_df.count())
print("album ratings :", train_album_df.count())
print("artist ratings:", train_artist_df.count())
print("genre ratings :", train_genre_df.count())


track ratings : 5480041
album ratings : 2386505
artist ratings: 3829295
genre ratings : 707734


In [7]:
train_track_df = train_track_df.repartition(64, "userID").cache()
train_album_df = train_album_df.repartition(64, "userID").cache()
train_artist_df = train_artist_df.repartition(64, "userID").cache()
train_genre_df = train_genre_df.repartition(64, "userID").cache()

print("track rows :", train_track_df.count())
print("album rows :", train_album_df.count())
print("artist rows:", train_artist_df.count())
print("genre rows :", train_genre_df.count())


track rows : 5480041
album rows : 2386505
artist rows: 3829295
genre rows : 707734


In [8]:
# Cell 6: Helper to train ALS
def train_als(
    df,
    rank=32,
    maxIter=8,
    regParam=0.1,
    userCol="userID",
    itemCol="itemID",
    ratingCol="rating",
    nonnegative=True,
    coldStartStrategy="nan"
):
    als = ALS(
        rank=rank,
        maxIter=maxIter,
        regParam=regParam,
        userCol=userCol,
        itemCol=itemCol,
        ratingCol=ratingCol,
        nonnegative=nonnegative,
        implicitPrefs=False,
        coldStartStrategy=coldStartStrategy
    )
    return als.fit(df)

In [9]:
# Cell 7: Train separate ALS models


# Track model: sparse, so slightly stronger regularization
t0 = time.time()
track_model = train_als(train_track_df, rank=32, maxIter=8, regParam=0.12)
print("track_model done in", round(time.time() - t0, 2), "sec")

# Album model: often strong signal, moderately dense
t0 = time.time()
album_model = train_als(train_album_df, rank=24, maxIter=8, regParam=0.10)
print("album_model done in", round(time.time() - t0, 2), "sec")

# Artist model: usually denser than track
t0 = time.time()
artist_model = train_als(train_artist_df, rank=20, maxIter=8, regParam=0.08)
print("artist_model done in", round(time.time() - t0, 2), "sec")

# Genre model: low-cardinality and broad; smaller latent space is often enough
t0 = time.time()
genre_model = train_als(train_genre_df, rank=12, maxIter=6, regParam=0.06)
print("genre_model done in", round(time.time() - t0, 2), "sec")

print("All ALS models trained.")


track_model done in 63.14 sec
album_model done in 19.31 sec
artist_model done in 20.46 sec
genre_model done in 5.73 sec
All ALS models trained.


In [10]:
# Cell 8: Prepare candidate tables for scoring

# Base test candidates
test_candidates_df = test_df.dropDuplicates(["userID", "trackID"])

# Track-level candidates for track ALS
track_candidates_df = (
    test_candidates_df
    .select(
        "userID",
        F.col("trackID").alias("itemID")
    )
)

# Album-level candidates for album ALS
album_candidates_df = (
    test_candidates_df
    .join(track_core_df, on="trackID", how="left")
    .filter(F.col("albumID").isNotNull())
    .select(
        "userID",
        "trackID",
        F.col("albumID").alias("itemID")
    )
)

# Artist-level candidates for artist ALS
artist_candidates_df = (
    test_candidates_df
    .join(track_core_df, on="trackID", how="left")
    .filter(F.col("artistID").isNotNull())
    .select(
        "userID",
        "trackID",
        F.col("artistID").alias("itemID")
    )
)

# Genre-level candidates for genre ALS
genre_candidates_df = (
    test_candidates_df
    .join(track_genres_df, on="trackID", how="left")
    .filter(F.col("genreID").isNotNull())
    .select(
        "userID",
        "trackID",
        F.col("genreID").alias("itemID")
    )
)

print("track candidate rows :", track_candidates_df.count())
print("album candidate rows :", album_candidates_df.count())
print("artist candidate rows:", artist_candidates_df.count())
print("genre candidate rows :", genre_candidates_df.count())


track candidate rows : 120000
album candidate rows : 111428
artist candidate rows: 109109
genre candidate rows : 412482


In [11]:
# Cell 9: Predict ALS scores for each entity space

def clean_prediction_col(df, pred_col_name):
    """
    Spark ALS may return NaN for cold-start rows.
    Convert NaN to null so adaptive weighting is easier.
    """
    return (
        df.withColumn(
            pred_col_name,
            F.when(F.isnan("prediction") | F.col("prediction").isNull(), F.lit(None))
             .otherwise(F.col("prediction"))
        )
        .drop("prediction")
    )

# Track predictions
track_pred_df = (
    track_model.transform(track_candidates_df)
)
track_pred_df = clean_prediction_col(track_pred_df, "track_als")
track_pred_df = track_pred_df.select("userID", F.col("itemID").alias("trackID"), "track_als")

# Album predictions
album_pred_df = (
    album_model.transform(album_candidates_df.select("userID", "itemID"))
    .join(album_candidates_df, on=["userID", "itemID"], how="inner")
)
album_pred_df = clean_prediction_col(album_pred_df, "album_als")
album_pred_df = album_pred_df.select("userID", "trackID", "album_als")

# Artist predictions
artist_pred_df = (
    artist_model.transform(artist_candidates_df.select("userID", "itemID"))
    .join(artist_candidates_df, on=["userID", "itemID"], how="inner")
)
artist_pred_df = clean_prediction_col(artist_pred_df, "artist_als")
artist_pred_df = artist_pred_df.select("userID", "trackID", "artist_als")

# Genre predictions: predict one row per (user, track, genre), then aggregate
genre_pred_long_df = (
    genre_model.transform(genre_candidates_df.select("userID", "itemID"))
    .join(genre_candidates_df, on=["userID", "itemID"], how="inner")
)
genre_pred_long_df = clean_prediction_col(genre_pred_long_df, "genre_als_raw")

genre_pred_df = (
    genre_pred_long_df
    .groupBy("userID", "trackID")
    .agg(
        F.avg("genre_als_raw").alias("genre_mean_als"),
        F.max("genre_als_raw").alias("genre_max_als")
    )
)

print("Prediction tables created.")


Prediction tables created.


In [12]:
# Cell 10: Combine all entity-level ALS predictions

candidate_scores_df = (
    test_candidates_df
    .join(track_pred_df, on=["userID", "trackID"], how="left")
    .join(album_pred_df, on=["userID", "trackID"], how="left")
    .join(artist_pred_df, on=["userID", "trackID"], how="left")
    .join(genre_pred_df, on=["userID", "trackID"], how="left")
)

candidate_scores_df.show(10, truncate=False)


+------+-------+---------+----------+----------+------------------+-------------+
|userID|trackID|track_als|album_als |artist_als|genre_mean_als    |genre_max_als|
+------+-------+---------+----------+----------+------------------+-------------+
|199862|80123  |55.015366|52.196342 |44.753838 |157.63624572753906|315.2725     |
|199884|5444   |47.07106 |64.47221  |46.35817  |84.40320260184151 |87.73044     |
|199921|95242  |103.03074|22.67155  |70.857155 |76.08656215667725 |120.2652     |
|199936|89110  |49.885494|96.663925 |53.841545 |NULL              |NULL         |
|199982|114059 |61.762524|95.33524  |92.0935   |115.43550109863281|115.4355     |
|200020|21102  |32.86882 |106.969055|78.540886 |138.08575897216798|348.91       |
|200083|66989  |77.51928 |74.24396  |40.841427 |103.72868251800537|194.04266    |
|200203|271107 |95.64653 |86.083374 |99.883255 |NULL              |NULL         |
|200243|159305 |29.371754|35.11923  |37.494656 |36.43064880371094 |36.43065     |
|200313|171859 |

In [13]:
# Cell 11A: Clean candidate score table

# Safety dedupe before any ranking logic
candidate_scores_df = (
    candidate_scores_df
    .groupBy("userID", "trackID")
    .agg(
        F.max("track_als").alias("track_als"),
        F.max("album_als").alias("album_als"),
        F.max("artist_als").alias("artist_als"),
        F.max("genre_mean_als").alias("genre_mean_als"),
        F.max("genre_max_als").alias("genre_max_als")
    )
)

print("Rows after safety dedupe:", candidate_scores_df.count())
candidate_scores_df.show(10, truncate=False)


Rows after safety dedupe: 120000
+------+-------+---------+---------+----------+------------------+-------------+
|userID|trackID|track_als|album_als|artist_als|genre_mean_als    |genre_max_als|
+------+-------+---------+---------+----------+------------------+-------------+
|199813|21571  |55.3654  |67.508896|83.58827  |211.76010131835938|211.7601     |
|199813|188441 |26.017437|57.025997|92.88547  |189.12236404418945|299.17026    |
|199814|52519  |114.94128|99.9117  |70.036194 |NULL              |NULL         |
|199817|83722  |106.33185|99.711555|98.26105  |NULL              |NULL         |
|199819|123424 |91.080956|0.0      |58.551952 |195.0059814453125 |195.00598    |
|199819|193003 |77.6544  |0.0      |69.88768  |74.79885482788086 |120.747734   |
|199822|33495  |83.19893 |63.463585|62.86955  |133.59672864278158|243.24945    |
|199822|110099 |56.938858|65.1736  |58.070316 |51.11436208089193 |52.38143     |
|199824|253265 |49.27505 |64.149185|50.830704 |108.46619033813477|170.09183 

In [14]:
# ============================================================
# Cell 11A: Safety dedupe before rank blending
# Place this right after candidate_scores_df is created
# ============================================================

candidate_scores_rank_df = (
    candidate_scores_df
    .groupBy("userID", "trackID")
    .agg(
        F.max("track_als").alias("track_als"),
        F.max("album_als").alias("album_als"),
        F.max("artist_als").alias("artist_als"),
        F.max("genre_mean_als").alias("genre_mean_als"),
        F.max("genre_max_als").alias("genre_max_als")
    )
)

print("Rows after rank-input dedupe:", candidate_scores_rank_df.count())

dup_check = (
    candidate_scores_rank_df
    .groupBy("userID", "trackID")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate user-track rows after dedupe:", dup_check.count())

candidate_scores_rank_df.show(10, truncate=False)


Rows after rank-input dedupe: 120000
Duplicate user-track rows after dedupe: 0
+------+-------+---------+---------+----------+------------------+-------------+
|userID|trackID|track_als|album_als|artist_als|genre_mean_als    |genre_max_als|
+------+-------+---------+---------+----------+------------------+-------------+
|199813|21571  |55.3654  |67.508896|83.58827  |211.76010131835938|211.7601     |
|199813|188441 |26.017437|57.025997|92.88547  |189.12236404418945|299.17026    |
|199814|52519  |114.94128|99.9117  |70.036194 |NULL              |NULL         |
|199817|83722  |106.33185|99.711555|98.26105  |NULL              |NULL         |
|199819|123424 |91.080956|0.0      |58.551952 |195.0059814453125 |195.00598    |
|199819|193003 |77.6544  |0.0      |69.88768  |74.79885482788086 |120.747734   |
|199822|33495  |83.19893 |63.463585|62.86955  |133.59672864278158|243.24945    |
|199822|110099 |56.938858|65.1736  |58.070316 |51.11436208089193 |52.38143     |
|199824|253265 |49.27505 |64.1

In [15]:
'''
# First Iteration
# Cell 11: Adaptive normalized blending

# Starting weights
W_TRACK  = 0.30
W_ALBUM  = 0.30
W_ARTIST = 0.25
W_GENRE  = 0.15

scored_df = (
    candidate_scores_df
    # ----- mean-genre blend -----
    .withColumn(
        "weighted_sum_mean",
        F.coalesce(F.col("track_als")      * F.lit(W_TRACK),  F.lit(0.0)) +
        F.coalesce(F.col("album_als")      * F.lit(W_ALBUM),  F.lit(0.0)) +
        F.coalesce(F.col("artist_als")     * F.lit(W_ARTIST), F.lit(0.0)) +
        F.coalesce(F.col("genre_mean_als") * F.lit(W_GENRE),  F.lit(0.0))
    )
    .withColumn(
        "weight_used_mean",
        F.when(F.col("track_als").isNotNull(),      F.lit(W_TRACK)).otherwise(F.lit(0.0)) +
        F.when(F.col("album_als").isNotNull(),      F.lit(W_ALBUM)).otherwise(F.lit(0.0)) +
        F.when(F.col("artist_als").isNotNull(),     F.lit(W_ARTIST)).otherwise(F.lit(0.0)) +
        F.when(F.col("genre_mean_als").isNotNull(), F.lit(W_GENRE)).otherwise(F.lit(0.0))
    )
    .withColumn(
        "final_score_mean_genre",
        F.when(F.col("weight_used_mean") > 0, F.col("weighted_sum_mean") / F.col("weight_used_mean"))
         .otherwise(F.lit(-1e9))   # ALS-only safety fallback: push fully unscorable rows to bottom
    )

    # ----- max-genre blend -----
    .withColumn(
        "weighted_sum_max",
        F.coalesce(F.col("track_als")     * F.lit(W_TRACK),  F.lit(0.0)) +
        F.coalesce(F.col("album_als")     * F.lit(W_ALBUM),  F.lit(0.0)) +
        F.coalesce(F.col("artist_als")    * F.lit(W_ARTIST), F.lit(0.0)) +
        F.coalesce(F.col("genre_max_als") * F.lit(W_GENRE),  F.lit(0.0))
    )
    .withColumn(
        "weight_used_max",
        F.when(F.col("track_als").isNotNull(),     F.lit(W_TRACK)).otherwise(F.lit(0.0)) +
        F.when(F.col("album_als").isNotNull(),     F.lit(W_ALBUM)).otherwise(F.lit(0.0)) +
        F.when(F.col("artist_als").isNotNull(),    F.lit(W_ARTIST)).otherwise(F.lit(0.0)) +
        F.when(F.col("genre_max_als").isNotNull(), F.lit(W_GENRE)).otherwise(F.lit(0.0))
    )
    .withColumn(
        "final_score_max_genre",
        F.when(F.col("weight_used_max") > 0, F.col("weighted_sum_max") / F.col("weight_used_max"))
         .otherwise(F.lit(-1e9))
    )
)

scored_df.select(
    "userID", "trackID", "track_als", "album_als", "artist_als",
    "genre_mean_als", "genre_max_als",
    "final_score_mean_genre", "final_score_max_genre"
).show(10, truncate=False)
'''

'''
# Cell 11B: Per-user rank blending
# Replace raw score blending with this

# We convert each model output into a within-user rank score.
# Higher ALS prediction -> better rank.
#
# Because each user has only 6 candidate tracks, this is a very
# natural fit for the assignment and avoids cross-model scale issues.

# ============================================================
# Cell 11B: Per-user rank blending
# ============================================================

W_TRACK_RANK  = 0.45
W_ALBUM_RANK  = 0.30
W_ARTIST_RANK = 0.20
W_GENRE_RANK  = 0.05

w_track  = Window.partitionBy("userID").orderBy(F.col("track_als").desc_nulls_last(),      F.col("trackID").asc())
w_album  = Window.partitionBy("userID").orderBy(F.col("album_als").desc_nulls_last(),      F.col("trackID").asc())
w_artist = Window.partitionBy("userID").orderBy(F.col("artist_als").desc_nulls_last(),     F.col("trackID").asc())
w_genre  = Window.partitionBy("userID").orderBy(F.col("genre_mean_als").desc_nulls_last(), F.col("trackID").asc())

rank_blend_df = (
    candidate_scores_rank_df
    .withColumn("track_rank",  F.when(F.col("track_als").isNotNull(),      F.row_number().over(w_track)))
    .withColumn("album_rank",  F.when(F.col("album_als").isNotNull(),      F.row_number().over(w_album)))
    .withColumn("artist_rank", F.when(F.col("artist_als").isNotNull(),     F.row_number().over(w_artist)))
    .withColumn("genre_rank",  F.when(F.col("genre_mean_als").isNotNull(), F.row_number().over(w_genre)))

    .withColumn("track_rank_score",  F.when(F.col("track_rank").isNotNull(),  F.lit(7) - F.col("track_rank")))
    .withColumn("album_rank_score",  F.when(F.col("album_rank").isNotNull(),  F.lit(7) - F.col("album_rank")))
    .withColumn("artist_rank_score", F.when(F.col("artist_rank").isNotNull(), F.lit(7) - F.col("artist_rank")))
    .withColumn("genre_rank_score",  F.when(F.col("genre_rank").isNotNull(),  F.lit(7) - F.col("genre_rank")))

    .withColumn(
        "rank_weighted_sum",
        F.coalesce(F.col("track_rank_score")  * F.lit(W_TRACK_RANK),  F.lit(0.0)) +
        F.coalesce(F.col("album_rank_score")  * F.lit(W_ALBUM_RANK),  F.lit(0.0)) +
        F.coalesce(F.col("artist_rank_score") * F.lit(W_ARTIST_RANK), F.lit(0.0)) +
        F.coalesce(F.col("genre_rank_score")  * F.lit(W_GENRE_RANK),  F.lit(0.0))
    )
    .withColumn(
        "rank_weight_used",
        F.when(F.col("track_rank_score").isNotNull(),  F.lit(W_TRACK_RANK)).otherwise(F.lit(0.0)) +
        F.when(F.col("album_rank_score").isNotNull(),  F.lit(W_ALBUM_RANK)).otherwise(F.lit(0.0)) +
        F.when(F.col("artist_rank_score").isNotNull(), F.lit(W_ARTIST_RANK)).otherwise(F.lit(0.0)) +
        F.when(F.col("genre_rank_score").isNotNull(),  F.lit(W_GENRE_RANK)).otherwise(F.lit(0.0))
    )
    .withColumn(
        "final_rank_blend_score",
        F.when(F.col("rank_weight_used") > 0, F.col("rank_weighted_sum") / F.col("rank_weight_used"))
         .otherwise(F.lit(-1e9))
    )
)

rank_blend_df.select(
    "userID", "trackID",
    "track_als", "album_als", "artist_als", "genre_mean_als",
    "track_rank", "album_rank", "artist_rank", "genre_rank",
    "track_rank_score", "album_rank_score", "artist_rank_score", "genre_rank_score",
    "final_rank_blend_score"
).show(20, truncate=False)
'''

# ============================================================
# Cell 11C: Clipped + normalized score blending
# Put this after candidate_scores_rank_df is created
# ============================================================

# We clip ALS outputs into a rating-like range first because the
# raw model scales are not directly comparable across track/album/artist/genre.
# Then we normalize within each user's 6 candidates so each model
# contributes on the same 0-1 scale.

clipped_df = (
    candidate_scores_rank_df
    .withColumn(
        "track_clip",
        F.when(F.col("track_als").isNotNull(),
               F.least(F.lit(100.0), F.greatest(F.lit(0.0), F.col("track_als"))))
    )
    .withColumn(
        "album_clip",
        F.when(F.col("album_als").isNotNull(),
               F.least(F.lit(100.0), F.greatest(F.lit(0.0), F.col("album_als"))))
    )
    .withColumn(
        "artist_clip",
        F.when(F.col("artist_als").isNotNull(),
               F.least(F.lit(100.0), F.greatest(F.lit(0.0), F.col("artist_als"))))
    )
    .withColumn(
        "genre_mean_clip",
        F.when(F.col("genre_mean_als").isNotNull(),
               F.least(F.lit(100.0), F.greatest(F.lit(0.0), F.col("genre_mean_als"))))
    )
)

# Min-max normalize within each user across the 6 candidate tracks.
# If all values are identical for a component, we assign 0.5 so that
# model neither strongly helps nor hurts the candidate ordering.
w_user = Window.partitionBy("userID")

norm_df = (
    clipped_df
    .withColumn("track_min", F.min("track_clip").over(w_user))
    .withColumn("track_max", F.max("track_clip").over(w_user))
    .withColumn("album_min", F.min("album_clip").over(w_user))
    .withColumn("album_max", F.max("album_clip").over(w_user))
    .withColumn("artist_min", F.min("artist_clip").over(w_user))
    .withColumn("artist_max", F.max("artist_clip").over(w_user))
    .withColumn("genre_min", F.min("genre_mean_clip").over(w_user))
    .withColumn("genre_max", F.max("genre_mean_clip").over(w_user))

    .withColumn(
        "track_norm",
        F.when(F.col("track_clip").isNull(), None)
         .when(F.col("track_max") > F.col("track_min"),
               (F.col("track_clip") - F.col("track_min")) / (F.col("track_max") - F.col("track_min")))
         .otherwise(F.lit(0.5))
    )
    .withColumn(
        "album_norm",
        F.when(F.col("album_clip").isNull(), None)
         .when(F.col("album_max") > F.col("album_min"),
               (F.col("album_clip") - F.col("album_min")) / (F.col("album_max") - F.col("album_min")))
         .otherwise(F.lit(0.5))
    )
    .withColumn(
        "artist_norm",
        F.when(F.col("artist_clip").isNull(), None)
         .when(F.col("artist_max") > F.col("artist_min"),
               (F.col("artist_clip") - F.col("artist_min")) / (F.col("artist_max") - F.col("artist_min")))
         .otherwise(F.lit(0.5))
    )
    .withColumn(
        "genre_norm",
        F.when(F.col("genre_mean_clip").isNull(), None)
         .when(F.col("genre_max") > F.col("genre_min"),
               (F.col("genre_mean_clip") - F.col("genre_min")) / (F.col("genre_max") - F.col("genre_min")))
         .otherwise(F.lit(0.5))
    )
)

# Start with album/track focused weights because your earlier results
# suggest genre is the noisiest signal.
W_TRACK  = 0.45
W_ALBUM  = 0.30
W_ARTIST = 0.20
W_GENRE  = 0.05

norm_blend_df = (
    norm_df
    .withColumn(
        "norm_weighted_sum",
        F.coalesce(F.col("track_norm")  * F.lit(W_TRACK),  F.lit(0.0)) +
        F.coalesce(F.col("album_norm")  * F.lit(W_ALBUM),  F.lit(0.0)) +
        F.coalesce(F.col("artist_norm") * F.lit(W_ARTIST), F.lit(0.0)) +
        F.coalesce(F.col("genre_norm")  * F.lit(W_GENRE),  F.lit(0.0))
    )
    .withColumn(
        "norm_weight_used",
        F.when(F.col("track_norm").isNotNull(),  F.lit(W_TRACK)).otherwise(F.lit(0.0)) +
        F.when(F.col("album_norm").isNotNull(),  F.lit(W_ALBUM)).otherwise(F.lit(0.0)) +
        F.when(F.col("artist_norm").isNotNull(), F.lit(W_ARTIST)).otherwise(F.lit(0.0)) +
        F.when(F.col("genre_norm").isNotNull(),  F.lit(W_GENRE)).otherwise(F.lit(0.0))
    )
    .withColumn(
        "final_norm_blend_score",
        F.when(F.col("norm_weight_used") > 0, F.col("norm_weighted_sum") / F.col("norm_weight_used"))
         .otherwise(F.lit(-1e9))
    )
)

norm_blend_df.select(
    "userID", "trackID",
    "track_clip", "album_clip", "artist_clip", "genre_mean_clip",
    "track_norm", "album_norm", "artist_norm", "genre_norm",
    "final_norm_blend_score"
).show(20, truncate=False)


+------+-------+------------------+------------------+------------------+------------------+-------------------+-------------------+-------------------+-------------------+----------------------+
|userID|trackID|track_clip        |album_clip        |artist_clip       |genre_mean_clip   |track_norm         |album_norm         |artist_norm        |genre_norm         |final_norm_blend_score|
+------+-------+------------------+------------------+------------------+------------------+-------------------+-------------------+-------------------+-------------------+----------------------+
|199812|29189  |93.82508850097656 |89.60669708251953 |98.93761444091797 |79.89775784810384 |0.8792450048903536 |0.6083727733507278 |0.0                |1.0                |0.6281720842058776    |
|199812|142408 |100.0             |99.99742126464844 |99.5790023803711  |78.7029032389323  |1.0                |0.9999028313730474 |0.6037242637290035 |0.9823119265039605 |0.9198312984829129    |
|199812|211361 |98.9

In [ ]:
# Cell 12B: Generate submission from normalized blend

score_col = "final_norm_blend_score"

submission_norm_df = (
    norm_blend_df
    .withColumn(
        "final_rank",
        F.row_number().over(
            Window.partitionBy("userID").orderBy(F.col(score_col).desc(), F.col("trackID").asc())
        )
    )
    .withColumn("Predictor", F.when(F.col("final_rank") <= 3, 1).otherwise(0))
    .withColumn("TrackID", F.concat_ws("_", F.col("userID"), F.col("trackID")))
    .select("TrackID", "Predictor")
)

submission_norm_df.show(12, truncate=False)
print("Submission row count:", submission_norm_df.count())

dup_df = (
    submission_norm_df
    .groupBy("TrackID")
    .count()
    .filter(F.col("count") > 1)
)
print("Duplicate TrackID rows:", dup_df.count())

valid_df = (
    submission_norm_df
    .withColumn("uid", F.split(F.col("TrackID"), "_").getItem(0))
    .groupBy("uid")
    .agg(F.sum("Predictor").alias("num_ones"))
)
bad_users = valid_df.filter(F.col("num_ones") != 3)
print("Users without exactly 3 ones:", bad_users.count())

submission_norm_pd = submission_norm_df.toPandas()
submission_norm_pd.to_csv("submission_multi_als_normalized_blend.csv", index=False)
print("Saved: submission_multi_als_normalized_blend.csv")


+-------------+---------+
|TrackID      |Predictor|
+-------------+---------+
|199812_223706|1        |
|199812_142408|1        |
|199812_130023|1        |
|199812_211361|0        |
|199812_29189 |0        |
|199812_276940|0        |
|199827_184197|1        |
|199827_197936|1        |
|199827_183841|1        |
|199827_75369 |0        |
|199827_92599 |0        |
|199827_138391|0        |
+-------------+---------+
only showing top 12 rows

Submission row count: 120000
Duplicate TrackID rows: 0
Users without exactly 3 ones: 0
Saved: submission_multi_als_normalized_blend.csv


In [20]:
# Cell 14: Diagnostics for normalized blend

diagnostics_df = norm_blend_df.select(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("track_als").isNotNull(), 1).otherwise(0)).alias("track_available"),
    F.sum(F.when(F.col("album_als").isNotNull(), 1).otherwise(0)).alias("album_available"),
    F.sum(F.when(F.col("artist_als").isNotNull(), 1).otherwise(0)).alias("artist_available"),
    F.sum(F.when(F.col("genre_mean_als").isNotNull(), 1).otherwise(0)).alias("genre_available"),

    F.sum(F.when(F.col("track_norm").isNotNull(), 1).otherwise(0)).alias("track_norm_available"),
    F.sum(F.when(F.col("album_norm").isNotNull(), 1).otherwise(0)).alias("album_norm_available"),
    F.sum(F.when(F.col("artist_norm").isNotNull(), 1).otherwise(0)).alias("artist_norm_available"),
    F.sum(F.when(F.col("genre_norm").isNotNull(), 1).otherwise(0)).alias("genre_norm_available"),

    F.sum(F.when(F.col("norm_weight_used") == 0, 1).otherwise(0)).alias("fully_missing_rows")
)

diagnostics_df.show(truncate=False)


+------+---------------+---------------+----------------+---------------+--------------------+--------------------+---------------------+--------------------+------------------+
|rows  |track_available|album_available|artist_available|genre_available|track_norm_available|album_norm_available|artist_norm_available|genre_norm_available|fully_missing_rows|
+------+---------------+---------------+----------------+---------------+--------------------+--------------------+---------------------+--------------------+------------------+
|120000|119974         |103458         |108535          |100351         |119974              |103458              |108535               |100351              |0                 |
+------+---------------+---------------+----------------+---------------+--------------------+--------------------+---------------------+--------------------+------------------+



In [22]:
# Cell 15: Try different normalized blend settings

blend_configs = [
    ("track_heavy",      0.45, 0.25, 0.20, 0.10),
    ("album_heavy",      0.25, 0.35, 0.25, 0.15),
    ("balanced",         0.30, 0.30, 0.25, 0.15),
    ("no_track",         0.00, 0.45, 0.35, 0.20),
    ("no_genre",         0.35, 0.35, 0.30, 0.00),
    ("track_album_only", 0.60, 0.40, 0.00, 0.00),
    ("track_album_art",  0.50, 0.30, 0.20, 0.00),
]

for name, wt, wa, wr, wg in blend_configs:
    temp_df = (
        norm_df
        .withColumn(
            "weighted_sum",
            F.coalesce(F.col("track_norm")  * F.lit(wt), F.lit(0.0)) +
            F.coalesce(F.col("album_norm")  * F.lit(wa), F.lit(0.0)) +
            F.coalesce(F.col("artist_norm") * F.lit(wr), F.lit(0.0)) +
            F.coalesce(F.col("genre_norm")  * F.lit(wg), F.lit(0.0))
        )
        .withColumn(
            "weight_used",
            F.when(F.col("track_norm").isNotNull(),  F.lit(wt)).otherwise(F.lit(0.0)) +
            F.when(F.col("album_norm").isNotNull(),  F.lit(wa)).otherwise(F.lit(0.0)) +
            F.when(F.col("artist_norm").isNotNull(), F.lit(wr)).otherwise(F.lit(0.0)) +
            F.when(F.col("genre_norm").isNotNull(),  F.lit(wg)).otherwise(F.lit(0.0))
        )
        .withColumn(
            "final_score",
            F.when(F.col("weight_used") > 0, F.col("weighted_sum") / F.col("weight_used"))
             .otherwise(F.lit(-1e9))
        )
    )

    stats = temp_df.select(
        F.mean("final_score").alias("mean_score"),
        F.stddev("final_score").alias("std_score"),
        F.min("final_score").alias("min_score"),
        F.max("final_score").alias("max_score")
    ).collect()[0]

    print(
        f"{name:16s} | "
        f"mean={stats['mean_score']:.4f} | "
        f"std={stats['std_score']:.4f} | "
        f"min={stats['min_score']:.4f} | "
        f"max={stats['max_score']:.4f}"
    )


track_heavy      | mean=0.5437 | std=0.2625 | min=0.0000 | max=1.0000
album_heavy      | mean=0.5508 | std=0.2527 | min=0.0000 | max=1.0000
balanced         | mean=0.5500 | std=0.2509 | min=0.0000 | max=1.0000
no_track         | mean=-20524999.4507 | std=141788193.9020 | min=-1000000000.0000 | max=1.0000
no_genre         | mean=-16666.1296 | std=4082465.8963 | min=-1000000000.0000 | max=1.0000
track_album_only | mean=-33332.8008 | std=5773430.5251 | min=-1000000000.0000 | max=1.0000
track_album_art  | mean=-16666.1318 | std=4082465.8963 | min=-1000000000.0000 | max=1.0000


In [25]:
# Save trained ALS models
os.makedirs("SavedModels/multi_als_fallback", exist_ok=True)

norm_blend_df.select(
    "userID",
    "trackID",
    "final_norm_blend_score"
).toPandas().to_csv(
    "SavedModels/multi_als_fallback/final_norm_scores.csv",
    index=False
)

# Save Metadata:
track_core_df.toPandas().to_csv(
    "SavedModels/multi_als_fallback/track_core_df.csv",
    index=False
)

track_genres_df.toPandas().to_csv(
    "SavedModels/multi_als_fallback/track_genres_df.csv",
    index=False
)